# Phase 1 Ingest Smoke Test

Set `PDF_PATH` to a local PDF, then run the notebook to parse, embed, index, and retrieve text/image results.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "backend").exists() and (ROOT.parent / "backend").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PDF_PATH = Path("sample.pdf")
DOC_ID = "test-doc"
DOC_NAME = PDF_PATH.name
QUERY = "What are the main results shown in the figures?"

assert PDF_PATH.exists(), f"Set PDF_PATH to an existing PDF. Current value: {PDF_PATH}"

In [ ]:
from backend.services.pdf_parser import parse_pdf

parsed = parse_pdf(PDF_PATH, doc_id=DOC_ID, doc_name=DOC_NAME, output_dir="uploads/images")
print(f"text chunks: {len(parsed.text_chunks)}")
print(f"table chunks: {len(parsed.table_chunks)}")
print(f"image chunks: {len(parsed.image_chunks)}")

In [ ]:
from backend.services.embedder import ClipEmbedder, TextEmbedder
from backend.services.vector_store import ChromaVectorStore

text_embedder = TextEmbedder()
clip_embedder = ClipEmbedder()
store = ChromaVectorStore(persist_dir="chroma_db")

text_like_chunks = parsed.text_chunks + parsed.table_chunks
text_embeddings = text_embedder.embed_texts([chunk.content for chunk in text_like_chunks])
image_embeddings = clip_embedder.embed_images([chunk.content for chunk in parsed.image_chunks])

store.add_text_chunks(text_like_chunks, text_embeddings)
store.add_image_chunks(parsed.image_chunks, image_embeddings)
print("Indexed PDF content into ChromaDB.")

In [ ]:
text_query_embedding = text_embedder.embed_text(QUERY)
image_query_embedding = clip_embedder.embed_query_for_images(QUERY)

text_results = store.query_text(text_query_embedding, top_k=5, doc_ids=[DOC_ID])
image_results = store.query_images(image_query_embedding, top_k=5, doc_ids=[DOC_ID])

print("TEXT RESULTS")
for result in text_results:
    metadata = result["metadata"]
    print({"id": result["id"], "score": result["score"], "page": metadata.get("page"), "type": metadata.get("type")})
    print((metadata.get("content") or "")[:500])
    print("---")

print("IMAGE RESULTS")
for result in image_results:
    metadata = result["metadata"]
    print({"id": result["id"], "score": result["score"], "page": metadata.get("page"), "path": metadata.get("content")})